In [5]:
import pandas as pd
import numpy as np
import re

# Load raw data
df = pd.read_csv('data/raw/nascar_2025_2026.csv')  # or my scraped file
print(f"Loaded {len(df)} rows")
print(f"Original columns: {df.columns.tolist()}")

Loaded 871 rows
Original columns: ['year', 'race_number', 'race_name', 'race_date', 'track', 'track_type', 'track_miles', 'total_laps', 'caution_flags', 'caution_laps', 'lead_changes', 'avg_speed_mph', 'pole_speed_mph', 'margin_of_victory', 'attendance', 'finish_pos', 'start_pos', 'car_number', 'driver', 'sponsor', 'owner', 'car_make', 'laps_completed', 'status', 'laps_led', 'points', 'loop_avg_speed', 'loop_green_flag_passes', 'loop_quality_passes', 'loop_fastest_lap', 'loop_driver_rating']


In [6]:
# Create a column mapping for consistent naming
# Adjust based on your actual column names
column_mapping = {
    # Common variations -> Standard names
    'Fin': 'Finish_Position',
    'finish_pos': 'Finish_Position',
    'fin': 'Finish_Position',
    'Pos': 'Finish_Position',
    'Position': 'Finish_Position',
    'St': 'Start_Position',
    'start_pos': 'Start_Position',
    'Start': 'Start_Position',
    'driver': 'Driver',
    'Team': 'Team',
    'owner': 'Owner',
    'sponsor': 'Sponsor',
    'points': 'Points',
    'race_date': 'Race_Date',
    'Laps': 'Laps_Completed',
    'Led': 'Laps_Led',
    'laps_led': 'Laps_Led',
    'laps_completed': 'Laps_Completed',
    'Race': 'Race_Name',
    'Track': 'Track',
    'Date': 'Race_Date'
}

# Rename columns that exist in our mapping
df = df.rename(columns={col: column_mapping[col]
                        for col in df.columns
                        if col in column_mapping})

print(f"Standardized columns: {df.columns.tolist()}")

Standardized columns: ['year', 'race_number', 'race_name', 'Race_Date', 'track', 'track_type', 'track_miles', 'total_laps', 'caution_flags', 'caution_laps', 'lead_changes', 'avg_speed_mph', 'pole_speed_mph', 'margin_of_victory', 'attendance', 'Finish_Position', 'Start_Position', 'car_number', 'Driver', 'Sponsor', 'Owner', 'car_make', 'Laps_Completed', 'status', 'Laps_Led', 'Points', 'loop_avg_speed', 'loop_green_flag_passes', 'loop_quality_passes', 'loop_fastest_lap', 'loop_driver_rating']


In [7]:
# Convert Finish_Position to numeric
# Handle non-numeric values (DNF, DNS, DQ may appear as text)
def clean_position(pos)-> int:
    """Convert position to integer, handling special cases."""
    if pd.isna(pos):
        return np.nan
    pos_str = str(pos).strip()
    # Extract just the number
    match = re.search(r'\d+', pos_str)
    if match:
        return int(match.group())
    return np.nan

df['Finish_Position'] = df['Finish_Position'].apply(clean_position)

# Same for Start_Position
if 'Start_Position' in df.columns:
    df['Start_Position'] = df['Start_Position'].apply(clean_position)
    df['Start_Position'] = pd.to_numeric(df['Start_Position'], errors='coerce').astype('Int64')
if 'Points' in df.columns:
    df['Points'] = pd.to_numeric(df['Points'], errors='coerce').astype('Int64')

# Convert Laps_Led to numeric
if 'Laps_Led' in df.columns:
    df['Laps_Led'] = pd.to_numeric(df['Laps_Led'], errors='coerce').fillna(0).astype(int)

# Convert Race_Date to datetime
if 'Race_Date' in df.columns:
    df['Race_Date'] = pd.to_datetime(df['Race_Date'], errors='coerce')

print("\nData types after conversion:")
print(df.dtypes)


Data types after conversion:
year                               int64
race_number                        int64
race_name                            str
Race_Date                 datetime64[us]
track                                str
track_type                           str
track_miles                      float64
total_laps                       float64
caution_flags                      int64
caution_laps                       int64
lead_changes                       int64
avg_speed_mph                    float64
pole_speed_mph                   float64
margin_of_victory                    str
attendance                       float64
Finish_Position                    int64
Start_Position                     Int64
car_number                         int64
Driver                               str
Sponsor                              str
Owner                                str
car_make                             str
Laps_Completed                     int64
status                     

In [8]:
# View unique driver names to identify inconsistencies
print("Unique drivers:", df['Driver'].nunique())
print("\nSample driver names:")
for name in sorted(df['Driver'].dropna().unique())[:20]:
    print(f"  '{name}'")
num_missing=(df['Driver'].isna().sum())
print(f"Missing {num_missing} drivers")

Unique drivers: 56

Sample driver names:
  'A.J. Allmendinger'
  'Alex Bowman'
  'Anthony Alfredo'
  'Austin Cindric'
  'Austin Dillon'
  'Austin Hill'
  'B.J. McLeod'
  'Brad Keselowski'
  'Bubba Wallace'
  'Carson Hocevar'
  'Casey Mears'
  'Chad Finchum'
  'Chase Briscoe'
  'Chase Elliott'
  'Chris Buescher'
  'Christopher Bell'
  'Cody Ware'
  'Cole Custer'
  'Connor Zilisch'
  'Corey Heim'
Missing 0 drivers


In [11]:
def clean_driver_name(name):
    """Standardize driver name format."""
    if pd.isna(name):
        return name

    # Remove extra whitespace
    name = ' '.join(str(name).split())

    # Remove common suffixes/prefixes
    name = re.sub(r'\s*\(i\)|\s*\*|\s*#\d+', '', name)

    # Capitalize properly
    name = name.title()

    return name.strip()

df['Driver'] = df['Driver'].apply(clean_driver_name)

# Check for common variations and standardize
driver_corrections = {
    # Add corrections as you find them, e.g.:
    # 'Denny M Hamlin': 'Denny Hamlin',
    # 'Hamlin, Denny': 'Denny Hamlin',
}

df['Driver'] = df['Driver'].replace(driver_corrections)

print(f"\nUnique drivers after cleaning: {df['Driver'].nunique()}")


Unique drivers after cleaning: 56


In [16]:
def clean_team_name(name):
    """Standardize team name format."""
    if pd.isna(name):
        return name

    name = ' '.join(str(name).split())

    # Common standardizations
    corrections = {
        'Joe Gibbs Racing': 'Joe Gibbs Racing',
        'JGR': 'Joe Gibbs Racing',
        'Hendrick Motorsports': 'Hendrick Motorsports',
        'HMS': 'Hendrick Motorsports',
        'Team Penske': 'Team Penske',
        'Penske': 'Team Penske',
        'Stewart-Haas Racing': 'Stewart-Haas Racing',
        'SHR': 'Stewart-Haas Racing',
        '23XI Racing': '23XI Racing',
        '23XI': '23XI Racing',
        'Richard Childress Racing': 'Richard Childress Racing',
        'RCR': 'Richard Childress Racing',
        'Trackhouse Racing': 'Trackhouse Racing',
    }

    for old, new in corrections.items():
        if old.lower() in name.lower():
            return new

    return name.strip()

df['Owner'] = df['Owner'].apply(clean_team_name)

print(f"Unique teams: {df['Owner'].nunique()}")
print("\nTeam list:")
for team in (df['Owner'].sort_values().unique()):
    print(f"  {team}")

Unique teams: 19

Team list:
  23XI Racing
  B.J. McLeod
  Beard Motorsports
  Bob Jenkins
  Carl Long
  Gene Haas
  HYAK Motorsports
  JR Motorsports
  Jack Roush
  Joe Gibbs
  Legacy Motor Club
  Matthew Kaulig
  Richard Childress
  Rick Hendrick
  Rick Ware
  Spire Motorsports
  Team Penske
  Trackhouse Racing
  Wood Brothers
  nan


In [23]:
# Create sponsor mapping for your target sponsors
# This maps driver+team combinations to their primary sponsor
# Based on your Project 1 selections

sponsor_mapping = {
    # Format: ('Driver Name', 'Team Name'): 'Primary Sponsor'

    # Example mappings for major 2024 sponsors:
    ('Denny Hamlin', 'Joe Gibbs'): 'FedEx',

    ('Austin Hill', 'Richard Childress'): 'cheddar\'s Scratch Kitchen',


    ('Todd Gilliland', 'Bob Jenkins'): 'Love\'s Travel Stops',
    ('Brad Keselowski', 'Jack Roush'): 'Castrol',

    ('Ross Chastain', 'Trackhouse Racing'): 'Busch Light',
    # Add more as needed based on your target sponsors
}

def get_sponsor(row):
    """Look up primary sponsor for driver/team combination."""
    key = (row['Driver'], row['Owner'])
    return sponsor_mapping.get(key, 'Other/Unknown')

df['Primary_Sponsor'] = df.apply(get_sponsor, axis=1)

# Check sponsor distribution
print("Sponsor distribution:")
print(df['Primary_Sponsor'].value_counts())

Sponsor distribution:
Primary_Sponsor
Other/Unknown                772
FedEx                         23
Castrol                       23
Love's Travel Stops           23
Busch Light                   23
cheddar's Scratch Kitchen      7
Name: count, dtype: int64


In [24]:
# Define your target sponsors
target_sponsors = [
    'FedEx',
    'Castrol',
    'Love\'s Travel Stops',
    'Busch Light',
    'cheddar\'s Scratch Kitchen',
    
]

# Create filtered dataset with just target sponsors
df_targets = df[df['Primary_Sponsor'].isin(target_sponsors)].copy()

print(f"\nFiltered to target sponsors: {len(df_targets)} rows")
print(df_targets['Primary_Sponsor'].value_counts())


Filtered to target sponsors: 99 rows
Primary_Sponsor
FedEx                        23
Castrol                      23
Love's Travel Stops          23
Busch Light                  23
cheddar's Scratch Kitchen     7
Name: count, dtype: int64


In [28]:
# Add race number for time series analysis
races_order = df.groupby('race_number')['year'].min().sort_values()
race_number_map = {race: i+1 for i, race in enumerate(races_order.index)}
df['race_number'] = df['race_number'].map(race_number_map)

# Calculate positions gained/lost
if 'Start_Position' in df.columns:
    df['Positions_Gained'] = df['Start_Position'] - df['Finish_Position']

print("Added derived columns:")
print(df[['race_number', 'Driver', 'Start_Position',
          'Finish_Position', 'Positions_Gained']].head(10))

Added derived columns:
   race_number            Driver  Start_Position  Finish_Position  \
0            1     William Byron               5                1   
1            1     Tyler Reddick              11                2   
2            1    Jimmie Johnson              40                3   
3            1     Chase Briscoe               1                4   
4            1  John H. Nemechek              18                5   
5            1       Alex Bowman              38                6   
6            1       Ryan Blaney              16                7   
7            1    Austin Cindric               2                8   
8            1   Justin Allgaier              19                9   
9            1    Chris Buescher               6               10   

   Positions_Gained  
0                 4  
1                 9  
2                37  
3                -3  
4                13  
5                32  
6                 9  
7                -6  
8                10